# SeasonSense: District Crop-Yield Forecasting for Rwanda
**NISR 2026 Big Data Hackathon · Track 1: Agricultural Productivity**

This notebook builds an end-to-end deep-learning pipeline that forecasts district-level crop yields (kg/ha) for each
agricultural season (A and B) by combining:

* **NISR Seasonal Agricultural Survey (SAS)** district tables for 2023A to 2026A (yields + farm-practice adoption), and
* **ERA5 daily weather** (Open-Meteo archive API) at each district centroid, aggregated into weekly in-season sequences.

The model is a **GRU weather encoder + entity embeddings** (district, crop, season) + a static-feature branch.
For this proposal stage, the model is trained for **one placeholder epoch** only, to prove the pipeline runs end to end.
All metrics are logged to **TensorBoard**.

In [1]:
# =====================================================================================================
# SECTION 1 - SETUP AND CONFIGURATION
# This cell imports every library the pipeline needs, fixes random seeds so the placeholder run is
# reproducible, and gathers all tunable settings into one CONFIG dictionary. Keeping hyperparameters
# in a single place means that the same values are used by the model, the training loop and the
# TensorBoard hparams logger, so what is reported is exactly what was run. The paths point to a local
# data/ folder (raw downloads are cached there so the notebook does not re-download on every run) and
# a runs/ folder where TensorBoard event files are written. The split is temporal: the five oldest
# seasons train the model, 2025B validates it and 2026A is held out as an unseen "future" test season.
# EPOCHS is deliberately set to 1 because this stage only has to prove that the pipeline runs end to
# end; full training and tuning happen later in the ML Pipeline course.
# =====================================================================================================
import os, re, json, time, random, warnings
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

warnings.filterwarnings("ignore")

CONFIG = {
    "seed": 42,
    "seasons": ["2023A", "2023B", "2024A", "2024B", "2025A", "2025B", "2026A"],
    "train_seasons": ["2023A", "2023B", "2024A", "2024B", "2025A"],
    "val_season": "2025B",
    "test_season": "2026A",
    "weeks": 22,              # length of the in-season weekly weather sequence
    "gru_hidden": 32,
    "emb_district": 6,
    "emb_crop": 6,
    "emb_season": 2,
    "static_hidden": 16,
    "head_hidden": 64,
    "dropout": 0.2,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 32,
    "epochs": 1,              # placeholder run only - NOT a fully trained model
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(CONFIG["seed"])
ROOT = Path.cwd()
RAW, PROC, RUNS = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "runs"
for p in (RAW, PROC, RUNS):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE}")

torch 2.14.1 | device: cpu


In [2]:
# =====================================================================================================
# SECTION 2 - DATA ACQUISITION (NISR SAS TABLES + DISTRICT BOUNDARIES)
# This cell downloads the official NISR Seasonal Agricultural Survey (SAS) Excel annex tables that
# NISR publishes openly on statistics.gov.rw, plus the open geoBoundaries ADM2 file containing the 30
# district polygons of Rwanda. The SAS annual tables for 2023, 2024 and 2025 each contain Season A and
# Season B district tables, and a separate workbook covers 2026 Season A. A small helper function,
# download(), is reused for every file: it skips files that are already cached in data/raw, so the
# pipeline can be re-run offline after the first execution. Only publicly downloadable sources are
# used, so anyone (including NISR reviewers) can reproduce the dataset from scratch without requesting
# microdata access. The cell finishes by listing the files and their sizes as a quick integrity check.
# =====================================================================================================
NISR = "https://statistics.gov.rw/sites/default/files/documents"
SOURCES = {
    "SAS_2023_annual_tables.xlsx": f"{NISR}/2024-10/SAS%202023_Annual%20Tables%20%283%29.xlsx",
    "SAS_2024_annual_tables.xlsx": f"{NISR}/2025-02/SAS%202024_Annual%20Tables.xlsx",
    "SAS_2025_annual_tables.xlsx": f"{NISR}/2025-12/SAS%202025_Annual%20Tables.xlsx",
    "SAS_2026A_tables.xlsx":       f"{NISR}/2026-04/Tables_2026%20Season%20A.xlsx",
    "rwanda_districts_adm2.geojson": "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/"
                                     "gbOpen/RWA/ADM2/geoBoundaries-RWA-ADM2_simplified.geojson",
}

def download(url, dest, timeout=120):
    """Download url to dest unless it is already cached."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    dest.write_bytes(r.content)
    return dest

for name, url in SOURCES.items():
    f = download(url, RAW / name)
    print(f"{name:<34} {f.stat().st_size/1024:8.1f} KB")

SAS_2023_annual_tables.xlsx           255.0 KB
SAS_2024_annual_tables.xlsx           337.2 KB
SAS_2025_annual_tables.xlsx           379.5 KB
SAS_2026A_tables.xlsx                 181.9 KB
rwanda_districts_adm2.geojson         282.3 KB


In [3]:
# =====================================================================================================
# SECTION 3 - PARSING THE NISR SAS EXCEL TABLES
# NISR's annex workbooks are formatted for printing rather than machine reading: titles sit in merged
# cells, headers span several rows, crop names change spelling between years ("Soya beans" in 2023 vs
# "Soybean" in 2026) and missing values are written as "-". This cell defines reusable functions that
# locate each table by matching its title text, detect which season (A/B and year) the table belongs
# to, find the 30 district rows by name, and harmonise crop names through an alias map. Group totals
# (e.g. "Cereals"), "Other crops", fruits and fodder are excluded because they mix unlike crops. The
# same parse_overall() function extracts the "Overall" percentage column from every farm-practice table
# (fertiliser, improved seed, pesticide and erosion control), so no logic is duplicated across topics.
# =====================================================================================================
DISTRICTS = ["Nyarugenge", "Gasabo", "Kicukiro", "Nyanza", "Gisagara", "Nyaruguru", "Huye", "Nyamagabe",
             "Ruhango", "Muhanga", "Kamonyi", "Karongi", "Rutsiro", "Rubavu", "Nyabihu", "Ngororero", "Rusizi",
             "Nyamasheke", "Rulindo", "Gakenke", "Musanze", "Burera", "Gicumbi", "Rwamagana", "Nyagatare",
             "Gatsibo", "Kayonza", "Kirehe", "Ngoma", "Bugesera"]
CROP_ALIASES = {
    "maize": "Maize", "sorghum": "Sorghum", "paddy rice": "Paddy rice", "wheat": "Wheat", "cassava": "Cassava",
    "sweet potato": "Sweet potato", "sweet potatoes": "Sweet potato", "irish potato": "Irish potato",
    "irish potatoes": "Irish potato", "yams & taro": "Yams & Taro", "cooking banana": "Cooking banana",
    "dessert banana": "Dessert banana", "banana for beer": "Banana for beer", "bush bean": "Bush bean",
    "climbing bean": "Climbing bean", "pea": "Pea", "peas": "Pea", "groundnuts": "Groundnuts",
    "ground nuts": "Groundnuts", "soybean": "Soybean", "soya beans": "Soybean", "vegetables": "Vegetables"}
TOPICS = {  # regex on the (lower-cased) table title -> output column
    "yield_kg_ha":         r"average yield by crop type (and|per) district",
    "inorganic_fert_pct":  r"use of inorganic fertili[sz]er",
    "organic_fert_pct":    r"use of organic fertili[sz]er",
    "improved_seed_pct":   r"use of (improved )?seeds? (types? )?by farmer",
    "pesticide_pct":       r"use of pesticides",
    "erosion_protect_pct": r"percentage of farmers (who practiced|by) agricultural practices",
}
PRACTICE_COLS = [k for k in TOPICS if k != "yield_kg_ha"]

def sheet_title(df):
    """First long string in the top rows of a sheet = the table title."""
    for v in df.iloc[:5].values.ravel():
        if isinstance(v, str) and len(v) > 15:
            return re.sub(r"\s+", " ", v)
    return ""

def parse_season(title):
    """'Season A 2025_...' or '2025 Season A_...' -> '2025A'."""
    m = re.search(r"season\s*([ab])\W*_?\s*(20\d\d)", title, re.I)
    if m:
        return f"{m.group(2)}{m.group(1).upper()}"
    m = re.search(r"(20\d\d)\s*season\s*([ab])(?![a-z])", title, re.I)
    return f"{m.group(1)}{m.group(2).upper()}" if m else None

def to_num(v):
    if isinstance(v, str):
        try:
            return float(v.strip())
        except ValueError:
            return np.nan      # '-', blanks, footnotes
    return np.nan if v is None or pd.isna(v) else float(v)

def district_rows(df):
    """Map district name -> (row index, column index of the name)."""
    found = {}
    for i, row in df.iterrows():
        for j, v in enumerate(row.values[:3]):
            if isinstance(v, str) and v.strip() in DISTRICTS:
                found[v.strip()] = (i, j)
    return found

def parse_yield(df, season):
    rows = district_rows(df)
    first = min(i for i, _ in rows.values())
    for i in range(first - 1, -1, -1):        # walk up from the first district row to the crop header
        header = [str(c).strip().lower() for c in df.iloc[i].values]
        if sum(c in CROP_ALIASES for c in header) >= 5:
            break
    out = []
    for d, (i, _) in rows.items():
        for j, c in enumerate(header):
            y = to_num(df.iat[i, j]) if c in CROP_ALIASES else np.nan
            if y > 0:   # 0 / '-' means the crop is not grown (or not measured) in that district
                out.append({"season": season, "district": d, "crop": CROP_ALIASES[c], "yield_kg_ha": y})
    return out

def parse_overall(df):
    """First numeric value right of the district name = the 'Overall' percentage."""
    res = {}
    for d, (i, j) in district_rows(df).items():
        nums = [x for x in map(to_num, df.iloc[i].values[j + 1:]) if not np.isnan(x)]
        res[d] = nums[0] if nums else np.nan
    return res

def parse_workbook(path):
    yields, practices = [], {}
    xl = pd.ExcelFile(path)
    for sheet in xl.sheet_names:
        df = xl.parse(sheet, header=None)
        title = sheet_title(df).lower()
        season = parse_season(title)
        if season is None or "large" in title or "site" in title:
            continue
        for col, pat in TOPICS.items():
            if re.search(pat, title):
                if col == "yield_kg_ha":
                    yields += parse_yield(df, season)
                else:
                    practices.setdefault((season, col), parse_overall(df))
    return yields, practices

all_yields, all_practices = [], {}
for f in [k for k in SOURCES if k.endswith(".xlsx")]:
    y, p = parse_workbook(RAW / f)
    all_yields += y
    for k, v in p.items():
        all_practices.setdefault(k, v)

yields = pd.DataFrame(all_yields).drop_duplicates(["season", "district", "crop"])
practices = (pd.DataFrame([{"season": s, "district": d, col: v}
                           for (s, col), m in all_practices.items() for d, v in m.items()])
             .groupby(["season", "district"]).first().reset_index())
print("yield rows per season:\n", yields.groupby("season").size().to_string())
print("\npractice table coverage (district-seasons):", practices.shape[0], "| missing:", int(practices.isna().sum().sum()))

yield rows per season:
 season
2023A    459
2023B    417
2024A    454
2024B    469
2025A    455
2025B    468
2026A    456

practice table coverage (district-seasons): 210 | missing: 0


In [4]:
# =====================================================================================================
# SECTION 4 - CLIMATE DATA: DISTRICT CENTROIDS AND DAILY ERA5 WEATHER
# Survey tables alone cannot explain why the same district's yield rises or falls between seasons;
# rainfall timing and heat during the season are the main drivers on Rwanda's mostly rain-fed farms
# (the SAS itself shows that only a small share of plots are irrigated). This cell computes the
# centroid of each district polygon with the shoelace formula, then requests daily precipitation,
# maximum and minimum temperature and FAO reference evapotranspiration (ET0) from the free Open-Meteo
# ERA5 archive API for all 30 centroids in a single call. The API also returns each point's elevation,
# which is kept as a static feature because altitude separates the cool northern highlands from the
# hot, dry eastern lowlands. The raw daily table is cached as a CSV so later runs work offline.
# =====================================================================================================
def polygon_centroid(coords):
    """Area-weighted centroid of a polygon ring list using the shoelace formula."""
    ring = np.asarray(coords[0])
    x, y = ring[:, 0], ring[:, 1]
    cross = x[:-1] * y[1:] - x[1:] * y[:-1]
    a = cross.sum() / 2
    return (((x[:-1] + x[1:]) * cross).sum() / (6 * a), ((y[:-1] + y[1:]) * cross).sum() / (6 * a))

def district_centroids(geojson_path):
    gj = json.loads(Path(geojson_path).read_text())
    rows = []
    for feat in gj["features"]:
        g = feat["geometry"]
        polys = [g["coordinates"]] if g["type"] == "Polygon" else g["coordinates"]
        biggest = max(polys, key=lambda p: len(p[0]))
        lon, lat = polygon_centroid(biggest)
        rows.append({"district": feat["properties"]["shapeName"], "lat": lat, "lon": lon})
    return pd.DataFrame(rows)

def fetch_weather(cent, start="2022-09-01", end="2026-02-28", cache=RAW / "era5_daily_districts.csv"):
    if cache.exists():
        return pd.read_csv(cache, parse_dates=["date"])
    params = {"latitude": ",".join(f"{v:.4f}" for v in cent.lat), "longitude": ",".join(f"{v:.4f}" for v in cent.lon),
              "start_date": start, "end_date": end, "timezone": "Africa/Kigali",
              "daily": "precipitation_sum,temperature_2m_max,temperature_2m_min,et0_fao_evapotranspiration"}
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=300)
    r.raise_for_status()
    frames = []
    for d, loc in zip(cent.district, r.json()):
        day = pd.DataFrame(loc["daily"]).rename(columns={"time": "date", "precipitation_sum": "precip",
                                                         "temperature_2m_max": "tmax", "temperature_2m_min": "tmin",
                                                         "et0_fao_evapotranspiration": "et0"})
        day["district"], day["elevation"] = d, loc["elevation"]
        frames.append(day)
    out = pd.concat(frames, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"])
    out.to_csv(cache, index=False)
    return out

centroids = district_centroids(RAW / "rwanda_districts_adm2.geojson")
assert set(centroids.district) == set(DISTRICTS), "district names differ between NISR and geoBoundaries"
weather = fetch_weather(centroids)
print(weather.shape, weather.date.min().date(), "->", weather.date.max().date())
weather.head()

(38310, 7) 2022-09-01 -> 2026-02-28


,date,precip,tmax,tmin,et0,district,elevation
0,2022-09-01,4.8,22.0,14.7,3.19,Nyarugenge,1744.0
1,2022-09-02,0.1,24.3,12.9,3.71,Nyarugenge,1744.0
2,2022-09-03,1.8,23.5,15.0,3.84,Nyarugenge,1744.0
3,2022-09-04,0.0,25.0,12.9,4.57,Nyarugenge,1744.0
4,2022-09-05,0.0,25.3,12.8,4.77,Nyarugenge,1744.0


In [5]:
# =====================================================================================================
# SECTION 5 - IN-SEASON WEEKLY WEATHER SEQUENCES
# This cell turns daily weather into one fixed-length sequence per district and season, which is the
# input the GRU encoder reads. Rwanda's Season A runs roughly from September planting to February
# harvest, and Season B from mid-February to July, so each season gets a 22-week window starting on
# 1 September (A) or 15 February (B). For each week the function computes total rainfall, mean
# maximum and minimum temperature, total ET0, the climatic water balance (rain minus ET0, a simple
# drought signal) and the number of dry days (<1 mm). Keeping the weekly order, instead of collapsing
# the season into a single average, lets the model learn that a dry spell at flowering hurts more than
# the same deficit at planting, and lets the same model produce early, partial-season forecasts.
# =====================================================================================================
WEATHER_FEATS = ["rain_mm", "tmax_c", "tmin_c", "et0_mm", "water_balance_mm", "dry_days"]

def season_start(season):
    year, s = int(season[:4]), season[-1]
    return date(year - 1, 9, 1) if s == "A" else date(year, 2, 15)

def weekly_sequence(daily, start, weeks):
    """Aggregate one district's daily weather into a (weeks x features) array."""
    win = daily[(daily.date >= pd.Timestamp(start)) &
                (daily.date < pd.Timestamp(start + timedelta(weeks=weeks)))].copy()
    win["week"] = (win.date - pd.Timestamp(start)).dt.days // 7
    g = win.groupby("week")
    seq = pd.DataFrame({"rain_mm": g.precip.sum(), "tmax_c": g.tmax.mean(), "tmin_c": g.tmin.mean(),
                        "et0_mm": g.et0.sum(), "water_balance_mm": g.precip.sum() - g.et0.sum(),
                        "dry_days": g.precip.apply(lambda s: (s < 1).sum())})
    return seq.reindex(range(weeks)).to_numpy(dtype=np.float32)

seqs = {}
for d, daily in weather.groupby("district"):
    for s in CONFIG["seasons"]:
        seqs[(d, s)] = weekly_sequence(daily, season_start(s), CONFIG["weeks"])
assert not any(np.isnan(v).any() for v in seqs.values()), "weather gap inside a season window"
elevation = weather.groupby("district").elevation.first()
season_rain = {k: v[:, 0].sum() for k, v in seqs.items()}
print(f"{len(seqs)} sequences of shape {next(iter(seqs.values())).shape}; "
      f"season rainfall range {min(season_rain.values()):.0f}-{max(season_rain.values()):.0f} mm")

210 sequences of shape (22, 6); season rainfall range 234-2428 mm


In [6]:
# =====================================================================================================
# SECTION 6 - FEATURE TABLE, TEMPORAL SPLIT AND SCALING
# Here the three sources are joined into one modelling table with one row per district, crop and
# season. The target is log(yield): crop yields span two orders of magnitude (beans about 600 kg/ha,
# cassava over 13,000 kg/ha), and the log scale turns that into comparable relative errors. The split
# is strictly temporal (train on 2023A-2025A, validate on 2025B, test on 2026A) so the evaluation mimics
# real forecasting, where the model never sees the season it predicts. Every scaler is fitted on the
# training rows only and then applied to validation and test rows, which prevents information from the
# future leaking into training. Categorical IDs are created for district, crop and season type, and a
# naive baseline (each district-crop pair's mean training yield) is computed for later comparison.
# =====================================================================================================
df = (yields.merge(practices, on=["season", "district"], how="left")
            .assign(elevation=lambda x: x.district.map(elevation),
                    season_type=lambda x: x.season.str[-1].map({"A": 0, "B": 1}),
                    log_yield=lambda x: np.log(x.yield_kg_ha)))
df["district_id"] = df.district.map({d: i for i, d in enumerate(DISTRICTS)})
CROPS = sorted(df.crop.unique())
df["crop_id"] = df.crop.map({c: i for i, c in enumerate(CROPS)})
STATIC_COLS = PRACTICE_COLS + ["elevation"]

split = {"train": df[df.season.isin(CONFIG["train_seasons"])],
         "val":   df[df.season == CONFIG["val_season"]],
         "test":  df[df.season == CONFIG["test_season"]]}

def fit_scaler(a):
    mu, sd = np.nanmean(a, axis=0), np.nanstd(a, axis=0)
    return mu, np.where(sd == 0, 1, sd)

train_seq = np.stack([seqs[(d, s)] for d, s in zip(split["train"].district, split["train"].season)])
SEQ_MU, SEQ_SD = fit_scaler(train_seq.reshape(-1, len(WEATHER_FEATS)))
STA_MU, STA_SD = fit_scaler(split["train"][STATIC_COLS].to_numpy())
Y_MU, Y_SD = split["train"].log_yield.mean(), split["train"].log_yield.std()

pair_mean = split["train"].groupby(["district", "crop"]).log_yield.mean()
crop_mean = split["train"].groupby("crop").log_yield.mean()
def baseline_log_pred(frame):
    """Naive benchmark: mean training log-yield of the same district-crop pair (crop mean if unseen)."""
    keys = list(zip(frame.district, frame.crop))
    return np.array([pair_mean.get(k, crop_mean[k[1]]) for k in keys])

for k, v in split.items():
    print(f"{k:<5} rows={len(v):5d} seasons={sorted(v.season.unique())}")
print(f"{len(CROPS)} crops, {df.district.nunique()} districts, static features: {STATIC_COLS}")

train rows= 2254 seasons=['2023A', '2023B', '2024A', '2024B', '2025A']
val   rows=  468 seasons=['2025B']
test  rows=  456 seasons=['2026A']
17 crops, 30 districts, static features: ['inorganic_fert_pct', 'organic_fert_pct', 'improved_seed_pct', 'pesticide_pct', 'erosion_protect_pct', 'elevation']


In [7]:
# =====================================================================================================
# SECTION 7 - PYTORCH DATASET AND DATALOADERS
# This cell wraps the modelling table in a PyTorch Dataset so that the training loop receives ready
# tensors in mini-batches. Each item returns five pieces: the scaled weekly weather sequence for that
# district-season (22 x 6), the scaled static vector (farm-practice adoption rates and elevation),
# the district, crop and season-type IDs for the embedding layers, and the standardised log-yield
# target. Any missing static value is filled with the training mean (0 after scaling), so the model
# never receives NaNs. A single make_loader() helper builds the train, validation and test loaders
# with identical preprocessing, shuffling only the training set, which avoids copy-pasting the same
# construction code three times and guarantees every split is transformed in exactly the same way.
# =====================================================================================================
class YieldDataset(Dataset):
    def __init__(self, frame):
        self.seq = torch.tensor(np.stack([(seqs[(d, s)] - SEQ_MU) / SEQ_SD
                                          for d, s in zip(frame.district, frame.season)]), dtype=torch.float32)
        sta = (frame[STATIC_COLS].to_numpy() - STA_MU) / STA_SD
        self.sta = torch.tensor(np.nan_to_num(sta), dtype=torch.float32)
        self.cat = torch.tensor(frame[["district_id", "crop_id", "season_type"]].to_numpy(), dtype=torch.long)
        self.y = torch.tensor(((frame.log_yield - Y_MU) / Y_SD).to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.seq[i], self.sta[i], self.cat[i], self.y[i]

def make_loader(frame, shuffle):
    return DataLoader(YieldDataset(frame), batch_size=CONFIG["batch_size"], shuffle=shuffle)

loaders = {k: make_loader(v, shuffle=(k == "train")) for k, v in split.items()}
seq_b, sta_b, cat_b, y_b = next(iter(loaders["train"]))
print("batch shapes -> weather:", tuple(seq_b.shape), "static:", tuple(sta_b.shape),
      "ids:", tuple(cat_b.shape), "target:", tuple(y_b.shape))

batch shapes -> weather: (32, 22, 6) static: (32, 6) ids: (32, 3) target: (32,)


In [8]:
# =====================================================================================================
# SECTION 8 - MODEL ARCHITECTURE: GRU WEATHER ENCODER + ENTITY EMBEDDINGS
# This cell defines SeasonSenseNet, a hybrid network with three branches. (1) A single-layer GRU reads
# the 22-week weather sequence and summarises in-season conditions in its final hidden state; a GRU
# is used rather than an LSTM because it has fewer parameters, which matters with about 2,300 training
# rows. (2) Learned embeddings for district, crop and season type let crops and districts with few
# observations borrow statistical strength from similar ones, instead of 17 x 30 separate models.
# (3) A small dense layer encodes the static farm-practice and elevation features. The three
# representations are concatenated and passed through a regularised MLP head (dropout) that outputs
# the standardised log-yield. A printed parameter count confirms that the model is small enough for
# the dataset size and to train on a laptop CPU.
# =====================================================================================================
class SeasonSenseNet(nn.Module):
    def __init__(self, n_weather, n_static, n_districts, n_crops, cfg):
        super().__init__()
        self.gru = nn.GRU(n_weather, cfg["gru_hidden"], batch_first=True)
        self.emb_district = nn.Embedding(n_districts, cfg["emb_district"])
        self.emb_crop = nn.Embedding(n_crops, cfg["emb_crop"])
        self.emb_season = nn.Embedding(2, cfg["emb_season"])
        self.static = nn.Sequential(nn.Linear(n_static, cfg["static_hidden"]), nn.ReLU())
        joint = cfg["gru_hidden"] + cfg["static_hidden"] + cfg["emb_district"] + cfg["emb_crop"] + cfg["emb_season"]
        self.head = nn.Sequential(nn.Linear(joint, cfg["head_hidden"]), nn.ReLU(), nn.Dropout(cfg["dropout"]),
                                  nn.Linear(cfg["head_hidden"], 1))

    def forward(self, seq, static, cats):
        _, h = self.gru(seq)                                    # h: (1, batch, hidden)
        z = torch.cat([h[-1], self.static(static), self.emb_district(cats[:, 0]),
                       self.emb_crop(cats[:, 1]), self.emb_season(cats[:, 2])], dim=1)
        return self.head(z).squeeze(-1)

model = SeasonSenseNet(len(WEATHER_FEATS), len(STATIC_COLS), len(DISTRICTS), len(CROPS), CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"trainable parameters: {n_params:,}")

SeasonSenseNet(
  (gru): GRU(6, 32, batch_first=True)
  (emb_district): Embedding(30, 6)
  (emb_crop): Embedding(17, 6)
  (emb_season): Embedding(2, 2)
  (static): Sequential(
    (0): Linear(in_features=6, out_features=16, bias=True)
    (1): ReLU()
  )
  (head): Sequential(
    (0): Linear(in_features=62, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)
trainable parameters: 8,335


In [9]:
# =====================================================================================================
# SECTION 9 - TRAINING AND EVALUATION FUNCTIONS
# This cell defines the reusable building blocks of the training loop so that the same code serves
# training, validation and testing. train_one_epoch() runs mini-batch gradient descent with the Adam
# optimiser and gradient clipping, logging the loss of every batch to TensorBoard so that even a
# single placeholder epoch produces a visible learning curve. predict() runs the model in evaluation
# mode without gradients and converts outputs back from standardised log-yield to kg/ha. metrics()
# reports errors that are meaningful to agricultural planners: mean absolute percentage error (MAPE)
# and median APE, because yields differ hugely between crops, plus RMSE and R-squared on the log
# scale. The loss function is the Huber loss, which is less sensitive than MSE to the occasional
# extreme district-crop value produced by small survey samples.
# =====================================================================================================
def train_one_epoch(model, loader, optimiser, loss_fn, writer, epoch, global_step):
    model.train()
    total, n = 0.0, 0
    for seq, sta, cat, y in loader:
        seq, sta, cat, y = seq.to(DEVICE), sta.to(DEVICE), cat.to(DEVICE), y.to(DEVICE)
        optimiser.zero_grad()
        loss = loss_fn(model(seq, sta, cat), y)
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimiser.step()
        writer.add_scalar("batch/train_loss", loss.item(), global_step)
        writer.add_scalar("batch/grad_norm", grad_norm.item(), global_step)
        total, n, global_step = total + loss.item() * len(y), n + len(y), global_step + 1
    return total / n, global_step

@torch.no_grad()
def predict(model, loader, loss_fn):
    model.eval()
    preds, ys, total = [], [], 0.0
    for seq, sta, cat, y in loader:
        out = model(seq.to(DEVICE), sta.to(DEVICE), cat.to(DEVICE)).cpu()
        total += loss_fn(out, y).item() * len(y)
        preds.append(out); ys.append(y)
    p, y = torch.cat(preds).numpy() * Y_SD + Y_MU, torch.cat(ys).numpy() * Y_SD + Y_MU
    return p, y, total / len(y)          # log-yields (kg/ha) and mean loss

def metrics(log_pred, log_true):
    ape = np.abs(np.exp(log_pred) - np.exp(log_true)) / np.exp(log_true) * 100
    ss_res, ss_tot = ((log_true - log_pred) ** 2).sum(), ((log_true - log_true.mean()) ** 2).sum()
    return {"MAPE_pct": ape.mean(), "MedianAPE_pct": np.median(ape),
            "RMSE_log": np.sqrt(((log_true - log_pred) ** 2).mean()), "R2_log": 1 - ss_res / ss_tot}

In [10]:
# =====================================================================================================
# SECTION 10 - TENSORBOARD SET-UP AND PLACEHOLDER TRAINING RUN (1 EPOCH)
# This cell creates a time-stamped TensorBoard run directory and logs far more than the minimum. It
# writes the model computation graph (add_graph), every training batch's loss and gradient norm, the
# epoch-level training and validation loss, validation MAPE, median APE, RMSE and R-squared, the same
# metrics for the naive district-crop-average baseline, the learning rate, per-crop validation MAPE,
# weight histograms and the full hyperparameter set with its final metrics (HParams tab). The model is
# deliberately trained for only CONFIG["epochs"] = 1 epoch: the goal at proposal stage is to prove that
# data loading, the forward pass, back-propagation, evaluation and logging all run end to end, not to
# report final accuracy. Numbers printed here are therefore placeholder values from an untrained model.
# =====================================================================================================
def log_metrics(writer, prefix, m, step):
    for k, v in m.items():
        writer.add_scalar(f"{prefix}/{k}", v, step)

run_name = time.strftime("seasonsense_%Y%m%d-%H%M%S")
writer = SummaryWriter(log_dir=str(RUNS / run_name))
writer.add_graph(model, (seq_b.to(DEVICE), sta_b.to(DEVICE), cat_b.to(DEVICE)))
writer.add_text("config", "\n".join(f"{k}: {v}" for k, v in CONFIG.items()))

optimiser = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
loss_fn = nn.HuberLoss(delta=1.0)
base_val = metrics(baseline_log_pred(split["val"]), split["val"].log_yield.to_numpy())

# step 0 = untrained model, so the epoch curves show a start and an end point
p0, y0, l0 = predict(model, loaders["val"], loss_fn)
log_metrics(writer, "val", {"loss": l0, **metrics(p0, y0)}, 0)
log_metrics(writer, "baseline_val", base_val, 0)

global_step = 0
for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, global_step = train_one_epoch(model, loaders["train"], optimiser, loss_fn, writer, epoch, global_step)
    p_val, y_val, val_loss = predict(model, loaders["val"], loss_fn)
    val_m = metrics(p_val, y_val)
    writer.add_scalar("train/loss", train_loss, epoch)
    log_metrics(writer, "val", {"loss": val_loss, **val_m}, epoch)
    log_metrics(writer, "baseline_val", base_val, epoch)
    writer.add_scalar("train/learning_rate", optimiser.param_groups[0]["lr"], epoch)
    for name, prm in model.named_parameters():
        writer.add_histogram(f"weights/{name}", prm, epoch)
    vf = split["val"].assign(ape=np.abs(np.exp(p_val) / np.exp(y_val) - 1) * 100)
    for crop, ape in vf.groupby("crop").ape.mean().items():
        writer.add_scalar(f"val_MAPE_by_crop/{crop}", ape, epoch)
    print(f"epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
          f"val_MAPE={val_m['MAPE_pct']:.1f}% (baseline {base_val['MAPE_pct']:.1f}%) val_R2_log={val_m['R2_log']:.3f}")

writer.add_hparams({k: v for k, v in CONFIG.items() if isinstance(v, (int, float))},
                   {"hparam/val_MAPE": val_m["MAPE_pct"], "hparam/val_R2_log": val_m["R2_log"],
                    "hparam/val_loss": val_loss}, run_name="hparams")
writer.flush()
print("TensorBoard logs ->", RUNS / run_name)

epoch 1: train_loss=0.3806 val_loss=0.2915 val_MAPE=104.5% (baseline 15.4%) val_R2_log=0.412
TensorBoard logs -> /Users/nick/Documents/alu/nisr_competition/runs/seasonsense_20261008-130649


In [11]:
# =====================================================================================================
# SECTION 11 - HELD-OUT 2026A INFERENCE AND OUTPUT TABLE
# The last step proves that the pipeline produces usable output end to end: the placeholder model is
# applied to the unseen 2026 Season A test season, predictions are converted back to kg/ha and saved
# to data/processed/ as a CSV that a dashboard or app can read. The test metrics are logged to
# TensorBoard and printed next to the naive baseline. Because the model has been trained for only one
# epoch, these numbers are expected to be weak and are not the project's results; they only confirm
# that every stage from raw NISR Excel files to a forecast table works. The cell also shows how to
# open TensorBoard (in Colab or Jupyter: %load_ext tensorboard then %tensorboard --logdir runs) and
# closes the writer so all event files are fully written to disk.
# =====================================================================================================
p_test, y_test, test_loss = predict(model, loaders["test"], loss_fn)
test_m, base_test = metrics(p_test, y_test), metrics(baseline_log_pred(split["test"]), y_test)
log_metrics(writer, "test", {"loss": test_loss, **test_m}, CONFIG["epochs"])
writer.close()

out = split["test"][["season", "district", "crop", "yield_kg_ha"]].assign(
    predicted_kg_ha=np.exp(p_test).round(0), baseline_kg_ha=np.exp(baseline_log_pred(split["test"])).round(0))
out.to_csv(PROC / "forecast_2026A_placeholder.csv", index=False)
print(pd.DataFrame({"SeasonSenseNet (1 epoch)": test_m, "Naive baseline": base_test}).round(3))
print("\nRun:  tensorboard --logdir runs   (Colab/Jupyter: %load_ext tensorboard, %tensorboard --logdir runs)")
out.head(10)

               SeasonSenseNet (1 epoch)  Naive baseline
MAPE_pct                         89.458          13.702
MedianAPE_pct                    61.285          10.636
RMSE_log                          0.912           0.187
R2_log                            0.479           0.978

Run:  tensorboard --logdir runs   (Colab/Jupyter: %load_ext tensorboard, %tensorboard --logdir runs)


,season,district,crop,yield_kg_ha,predicted_kg_ha,baseline_kg_ha
2722,2026A,Nyarugenge,Maize,1299.848319,3097.0,1113.0
2723,2026A,Nyarugenge,Cassava,12061.092867,1671.0,11292.0
2724,2026A,Nyarugenge,Sweet potato,6290.083244,5829.0,5817.0
2725,2026A,Nyarugenge,Irish potato,2579.662393,5951.0,2664.0
2726,2026A,Nyarugenge,Yams & Taro,4157.535645,4075.0,6057.0
2727,2026A,Nyarugenge,Cooking banana,13388.758811,5083.0,13671.0
2728,2026A,Nyarugenge,Dessert banana,9745.118726,2881.0,9653.0
2729,2026A,Nyarugenge,Banana for beer,11720.199574,6985.0,10828.0
2730,2026A,Nyarugenge,Bush bean,518.724454,1634.0,519.0
2731,2026A,Nyarugenge,Climbing bean,1042.787070,987.0,710.0
